# Module 8.3 — LangGraph Fundamentals for RAG

LangGraph models RAG as a **state machine**:
- **State** — TypedDict carrying all intermediate data
- **Nodes** — functions that transform state
- **Edges** — transitions between nodes (fixed or conditional)

```
START → [retrieve] → [grade] →─ relevant ─→ [generate] → END
                            └─ irrelevant → [rewrite]  → [retrieve]
```

In [ ]:
from typing import TypedDict, List
from langgraph.graph import StateGraph, END
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain.schema import Document

# ── State ─────────────────────────────────────────────────────────────────────
class RAGState(TypedDict):
    question : str
    documents: List[Document]
    answer   : str
    grade    : str
    iterations: int

llm        = ChatOpenAI(model="gpt-4o-mini", temperature=0)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

docs = [
    Document(page_content="LangGraph is a library for building stateful, multi-actor applications with LLMs."),
    Document(page_content="In LangGraph, nodes are Python functions and edges define control flow."),
    Document(page_content="LangGraph supports cycles and conditional branching for complex agentic workflows."),
    Document(page_content="TypedDict is used to define the state schema in LangGraph applications."),
]
vs = Chroma.from_documents(docs, embeddings, collection_name="lg_demo")

# ── Nodes ─────────────────────────────────────────────────────────────────────
def retrieve_node(state: RAGState) -> RAGState:
    docs = vs.similarity_search(state["question"], k=3)
    return {**state, "documents": docs, "iterations": state.get("iterations", 0) + 1}

def grade_node(state: RAGState) -> RAGState:
    prompt = ChatPromptTemplate.from_template(
        "Is this document relevant to '{question}'? Answer only 'yes' or 'no'.\nDocument: {doc}"
    )
    grades = []
    for doc in state["documents"]:
        r = (prompt | llm | StrOutputParser()).invoke({"question": state["question"], "doc": doc.page_content})
        grades.append("yes" in r.lower())
    grade = "relevant" if any(grades) else "irrelevant"
    return {**state, "grade": grade}

def generate_node(state: RAGState) -> RAGState:
    ctx    = "\n".join(d.page_content for d in state["documents"])
    prompt = ChatPromptTemplate.from_template("Answer using context:\n{context}\n\nQ: {question}")
    answer = (prompt | llm | StrOutputParser()).invoke({"context": ctx, "question": state["question"]})
    return {**state, "answer": answer}

def rewrite_node(state: RAGState) -> RAGState:
    prompt   = ChatPromptTemplate.from_template("Rewrite for better retrieval: {question}")
    new_q    = (prompt | llm | StrOutputParser()).invoke({"question": state["question"]})
    return {**state, "question": new_q.strip()}

# ── Conditional routing ───────────────────────────────────────────────────────
def route_grade(state: RAGState) -> str:
    if state["grade"] == "relevant" or state["iterations"] >= 3:
        return "generate"
    return "rewrite"

# ── Build graph ───────────────────────────────────────────────────────────────
builder = StateGraph(RAGState)
builder.add_node("retrieve", retrieve_node)
builder.add_node("grade",    grade_node)
builder.add_node("generate", generate_node)
builder.add_node("rewrite",  rewrite_node)

builder.set_entry_point("retrieve")
builder.add_edge("retrieve", "grade")
builder.add_conditional_edges("grade", route_grade, {"generate": "generate", "rewrite": "rewrite"})
builder.add_edge("rewrite",  "retrieve")
builder.add_edge("generate", END)

rag_graph = builder.compile()

# ── Run ───────────────────────────────────────────────────────────────────────
result = rag_graph.invoke({
    "question": "How does LangGraph handle complex control flow?",
    "documents": [], "answer": "", "grade": "", "iterations": 0
})
print(f"Answer: {result['answer']}")
print(f"Iterations: {result['iterations']} | Grade: {result['grade']}")
